# PRM-as-a-Judge Quick Start

Run the repository's three real robot-rollout examples with the public Robo-Dopamine checkpoint, inspect the generated metrics and curves, and open the interactive report.

Use a **Python 3.10** kernel with an NVIDIA GPU. Run the cells from top to bottom. The dependency-install cell is optional when the environment is already prepared.

## 1. Locate the repository

The notebook can be launched from the repository root or from the `getting_started` directory.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'eval/run_eval.sh').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside a PRM-as-a-Judge checkout.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'eval') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'eval'))
print(f'Repository: {REPO_ROOT}')
print(f'Python: {sys.version.split()[0]} ({sys.executable})')

## 2. Install dependencies (optional)

Set `INSTALL_DEPENDENCIES = True` for a new Python 3.10 environment. Skip this cell when the Dopamine and notebook dependencies are already installed.

In [ ]:
import subprocess

INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    constraint = REPO_ROOT / 'constraints/dopamine-cu128-py310.txt'
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-e', '.[dopamine,notebook]',
        '-c', str(constraint),
    ], cwd=REPO_ROOT)
else:
    print('Dependency installation skipped.')

## 3. Configure the checkpoint, GPU, manifest, and output directory

`MODEL_SOURCE` may be either a Hugging Face model ID or an existing local checkpoint directory. You can also set the `PRM_CHECKPOINT` environment variable before launching Jupyter.

In [ ]:
# MODEL_SOURCE = os.environ.get(
#     'PRM_CHECKPOINT',
#     'tanhuajie2001/Robo-Dopamine-GRM-2.0-8B-Preview',
# )
MODEL_SOURCE = "/share/project/lyy/models/tanhuajie2001/Robo-Dopamine-GRM-2.0-8B-Preview"
GPU = os.environ.get('PRM_GPU', '0')
MANIFEST = REPO_ROOT / 'eval/examples/manifest_demo_cases.jsonl'
OUTPUT_ROOT = REPO_ROOT / 'eval/results/notebook_quickstart'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Model source: {MODEL_SOURCE}')
print(f'GPU: {GPU}')
print(f'Manifest: {MANIFEST}')
print(f'Output root: {OUTPUT_ROOT}')

## 4. Check CUDA and resolve the checkpoint

A Hugging Face model ID is downloaded to the standard local cache. A local directory is used directly.

In [ ]:
gpu_check = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_check.returncode != 0:
    raise RuntimeError('nvidia-smi failed. Run this notebook on a machine with an NVIDIA GPU.')
print(gpu_check.stdout.strip())

local_candidate = Path(MODEL_SOURCE).expanduser()
if local_candidate.is_dir():
    PRM_PATH = local_candidate.resolve()
else:
    from huggingface_hub import snapshot_download
    PRM_PATH = Path(snapshot_download(repo_id=MODEL_SOURCE)).resolve()
print(f'Checkpoint: {PRM_PATH}')

## 5. Preview and validate the three bundled cases

The manifest loader resolves media paths relative to the manifest file and checks the supported camera-field layout.

In [ ]:
from prm_judge.manifest import load_manifest

cases = load_manifest(MANIFEST)
assert len(cases) == 3, f'Expected 3 cases, found {len(cases)}'
preview = []
for case in cases:
    missing = [str(path) for path in case.videos.values() if not path.is_file()]
    if case.goal_image is not None and not case.goal_image.is_file():
        missing.append(str(case.goal_image))
    if missing:
        raise FileNotFoundError(f'{case.case_id}: missing {missing}')
    preview.append({
        'case_id': case.case_id,
        'task': case.task,
        'views': list(case.videos),
        'goal_image': str(case.goal_image) if case.goal_image else None,
    })
preview

## 6. Run the full three-case evaluation

This cell calls the repository's existing shell runner with explicit settings and enables report generation. Depending on the GPU, this can take several minutes.

In [ ]:
import time

started_at = time.time()
run_env = os.environ.copy()
run_env.update({
    'MANIFEST': str(MANIFEST),
    'PRM_PATH': str(PRM_PATH),
    'GPUS': GPU,
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'VISUALIZE': '1',
    'PYTHON': sys.executable,
})
subprocess.run(['bash', 'eval/run_eval.sh'], cwd=REPO_ROOT, env=run_env, check=True)

new_runs = [
    path for path in OUTPUT_ROOT.glob('run_*')
    if path.is_dir() and path.stat().st_mtime >= started_at - 2
]
if not new_runs:
    raise RuntimeError(f'No new run directory was created under {OUTPUT_ROOT}')
RUN_ROOT = max(new_runs, key=lambda path: path.stat().st_mtime)
print(f'Run root: {RUN_ROOT}')

## 7. Inspect the summary, core metrics, and case plots

In [ ]:
import csv
import json
from IPython.display import Image, Markdown, display

run_summary = json.loads((RUN_ROOT / 'run_summary.json').read_text(encoding='utf-8'))
display(Markdown('### Run summary'))
display(run_summary)

metrics_path = RUN_ROOT / 'visualizations/curve_metrics.csv'
with metrics_path.open(encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
display(Markdown('### Per-case curve metrics'))
display(metric_rows)

plots = sorted((RUN_ROOT / 'visualizations/cases').glob('*.png'))
display(Markdown(f'### Case plots ({len(plots)})'))
for plot in plots:
    display(Markdown(f'**{plot.stem}**'))
    display(Image(filename=str(plot), width=900))

## 8. Start the interactive report server

Re-running this cell closes the previous notebook-owned server before starting a new one. The server binds to loopback and chooses an available port automatically.

In [ ]:
import threading
from IPython.display import HTML, display
from prm_judge.serve import create_report_server

if '_report_server' in globals() and _report_server is not None:
    _report_server.shutdown()
    _report_server.server_close()
    if '_report_thread' in globals():
        _report_thread.join(timeout=5)

_report_server = create_report_server(RUN_ROOT, host='127.0.0.1', port=0)
_report_thread = threading.Thread(target=_report_server.serve_forever, daemon=True)
_report_thread.start()
report_port = _report_server.server_address[1]
report_url = f'http://127.0.0.1:{report_port}/report.html'
display(HTML(f'<a href="{report_url}" target="_blank">Open the full interactive report</a>'))
print(report_url)